# TA City — Phase 2 Morris FoM evaluation (R129–R194)

Implements raster and vector analysis using the TI-City model results and its spatial inputs.

### Authorship
@author: Alexandre Pereira Santos <br>
alexandre.santos(at)lmu.de<br>
- usespandas, geopandas, rasterio, rasterstats
- may include content from LLMs

### Features
1. Evaluates the 66 Morris pass-1 batch runs against  the observed 2000–2025 expansion, with the Phase 0 baseline anchors as the bar every candidate
must clear.

| Step | Cell | What it does |
|---|---|---|
| 1 | CONFIG | paths, scenario→run mapping, tolerances, seed sets |
| 2 | helpers | ESRI ASCII reader, run-raster discovery, ensemble aggregation |
| 3 | references | urban_early/late → `obs_new`, valid mask, demand check |
| 4 | run rasters | Morris s_m001–s_m066 (R129–R194, seed 11) + s0 (R125, seed 11) and the three null baselines (R126–R128, seeds 11/22/33) |
| 5 | evaluate | `evaluate_scenario_fom` per seed, results & baselines separately |
| 6 | ensemble | per-seed rows → per-scenario MEANS on both sides |
| 7 | skill | `normalize_against_baselines` → skill vs strongest baseline per metric |
| 8 | summary | anchor bar (s_edge, s0) and the Morris runs that clear it |
| 9 | y-metrics | per-run metric table for the later Morris sensitivity step |
| 10 | export | FoM / baseline / skill / y-metrics CSVs into docs/ |

Conventions (project-agreed): all module arguments explicit keyword
arguments; `per_tick='final'`, `variant='hits_only'`, `cell_km=0.150`,
`start_year=2000`.

Protocol notes (differs from Phase 0 on purpose):
- Morris batch runs are single-seed (11, common random numbers); s0 is
  re-evaluated at seed 11 here for a like-for-like comparison.
- Baselines (s_random, s_edge, s_roadbound) keep the Phase 0 protocol:
  ensemble means over seeds 11/22/33, so the anchor numbers match
  `ti_city_12_2` (s_edge strict FoM ≈ 0.335 is the bar; Phase 0 s0
  strict-FoM skill was ≈ −0.041).

Run *Restart & Run All* before saving; each cell is independently
re-runnable for troubleshooting (downstream cells depend on the `fom_*`
variables, not on printed state).

# 1. CONFIG 

In [11]:
from pathlib import Path

case_city = 'JAK' #MUM MAN JAK
drop_path = r'C:\\Sciebo\\05_GIS\\' # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\GIS' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\GIS'
external_data_path = Path(r'C:\\Sciebo\\00_data') # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\00_data' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\00_data'
git_path = Path(r'C:\\GIT\\ti-city-model') # adjust this to match the path on your computer, e.g. 'D:\\GIT\\ti-city-model' or 'C:\\Users\\x\\GIT\\ti-city-model'


scenario_run = {"s0": 125}                      # like-for-like seed-11 comparison
scenario_run.update({f"s_m{i:03d}": 128 + i for i in range(1, 67)})  # s_m001 -> R129 ... s_m066 -> R194
first_run = 129
last_run = 194

CONFIG = {
    "root": Path("../"),                       # path containing model/JAK and docs/
    "case_city": case_city,
    "demand": 29656,                           # frozen gross target
    "final_tick": 25,                          # 2000-2025, one tick per year
    "calibration_years": [2000, 2025],          # years to calibrate
    "cell_km": 0.150,
    "tolerances": (0, 1, 2, 3, 5),
    "variant": "hits_only",                    # 'hits_only' | 'full'
    "scenario_run": scenario_run,              # results runs (seed 11 only)
    "scenario_baseline": {                     # baseline runs (nulls, 3-seed ensemble)
        "s_random": 126, "s_edge": 127, "s_roadbound": 128},
    "seeds_results": (11,),                    # Morris batch: CRN, single seed
    "seeds_baseline": (11, 22, 33),            # Phase 0 protocol
}

jak     = CONFIG["root"] / "model" / case_city
docs    = CONFIG["root"] / "docs"
out_root = jak / "data" / "out"
in_dir   = jak / "data" / "in"
print(f"results: {len(CONFIG['scenario_run'])} scenarios "
      f"(seed {'/'.join(map(str, CONFIG['seeds_results']))}), "
      f"baselines: {len(CONFIG['scenario_baseline'])} "
      f"(seeds {'/'.join(map(str, CONFIG['seeds_baseline']))})")

results: 67 scenarios (seed 11), baselines: 3 (seeds 11/22/33)


# 2. helpers

In [2]:
# 2a. imports + validation module ---------------------------------------
import re
import sys
import numpy as np
import pandas as pd

# module import: adjust path if running from another folder
sys.path.insert(0, str((CONFIG["root"] / "notebooks").resolve()))
import ti_city_10_calibration_fuzzy_fom_validation_module as fom
from ti_city_10_calibration_fuzzy_fom_validation_module import (
    observed_new_from_rasters, restrict_mask,
    evaluate_scenario_fom, normalize_against_baselines)
print("module loaded:", fom.__name__)
git_path = Path(r'C:\GIT\ti-city-model') # adjust this to match the path on your computer, e.g. 'D:\GIT\ti-city-model' or 'C:\Users\x\GIT\ti-city-model'

module loaded: ti_city_10_calibration_fuzzy_fom_validation_module


In [3]:
# 2b. helpers: ESRI ASCII reader -----------------------------------------
def read_ascii_grid(path):
    """Read an ESRI ASCII grid (ncols/nrows/xllcorner/yllcorner/cellsize/
    NODATA_value header) into (array, header-dict). Row 0 = top/north."""
    path = Path(path)
    lines = path.read_text().splitlines()
    header = {}
    i = 0
    while i < len(lines) and len(header) < 6:
        parts = lines[i].split()
        if len(parts) == 2 and parts[0].lower() in (
                "ncols", "nrows", "xllcorner", "yllcorner", "cellsize",
                "nodata_value"):
            header[parts[0].lower()] = parts[1]
            i += 1
        else:
            break
    arr = np.loadtxt(lines[i:], dtype=float)
    nrows, ncols = int(header["nrows"]), int(header["ncols"])
    assert arr.shape == (nrows, ncols), f"{path.name}: {arr.shape} != ({nrows},{ncols})"
    return arr, header

# 2c. helpers: run-raster discovery + loading ----------------------------
def read_run_raster(run_dir, final_tick):
    """Find the final-tick export in a run folder. Preference: income-status
    exports (classes 1-4 new urban, 5 initial urban), then any expansion_*."""
    pattern = re.compile(r"t(\d+)\.asc$", re.IGNORECASE)
    candidates = [(int(m.group(1)), p) for p in Path(run_dir).glob("*.asc")
                  if (m := pattern.search(p.name))]
    assert candidates, f"No tick-stamped .asc exports in {run_dir}"
    max_tick, _ = max(candidates, key=lambda c: c[0])
    if max_tick != final_tick:
        print(f"WARNING: {run_dir} final export is t{max_tick:03d}, expected t{final_tick:03d}")
    same_tick = [p for t, p in candidates if t == max_tick]
    income = [p for p in same_tick if "income" in p.name.lower()]
    chosen = income[0] if income else same_tick[0]
    arr, _ = read_ascii_grid(chosen)
    print(f"  loaded {chosen.name}  <- {Path(run_dir).parent.name}/{Path(run_dir).name}")
    return arr

def load_run_dicts(mapping, seeds, label):
    """Build {scenario: {seed_key: array}} from the R-folder/seed structure.
    'seeds' is the seed tuple for THIS side (results vs baselines)."""
    results_dict = {}
    for scenario, run_num in mapping.items():
        series = {}
        for seed in seeds:
            run_dir = out_root / f"R{run_num}" / f"seed{seed}"
            series[f"seed{seed}"] = read_run_raster(run_dir, CONFIG["final_tick"])
        results_dict[scenario] = series
    print(f"loaded {label}: {len(results_dict)} scenarios "
          f"({sum(len(v) for v in results_dict.values())} rasters)")
    return results_dict

def aggregate_ensemble(fom_table):
    """Per-seed rows -> per-scenario ensemble MEANS (skill protocol)."""
    metric_cols = [c for c in fom_table.columns
                   if c.startswith("FuzzyFoM")] + ["FoM", "N_sim",
                   "Hits", "Misses", "FalseAlarms"]
    metric_cols = [c for c in metric_cols if c in fom_table.columns]
    agg = fom_table.groupby("Scenario")[metric_cols].mean().reset_index()
    n = fom_table.groupby("Scenario").size().rename("n_seeds").reset_index()
    return agg.merge(n, on="Scenario")

# 3. reference layers

In [4]:
# 3. reference layers: obs_new, valid mask ------------------------------
urban_early, h_early = read_ascii_grid(in_dir / "urban_early.asc")
urban_late,  _       = read_ascii_grid(in_dir / "urban_late.asc")
nodata = float(h_early.get("nodata_value", "-9999"))
valid_mask = (urban_early != nodata) & (urban_late != nodata)
urban_early_m = np.where(urban_early == nodata, 0, urban_early)
urban_late_m  = np.where(urban_late == nodata, 0, urban_late)
initial_urban = urban_early_m > 0

obs_all = observed_new_from_rasters(urban_late = urban_late_m,
                                    urban_early = urban_early_m)
obs_new = restrict_mask(obs_all, initial_urban = initial_urban,
                        valid_mask = valid_mask)
n_obs = int(np.sum(obs_new))
print(f"observed new-urban cells (AOI-masked): {n_obs:,}")
if abs(n_obs - CONFIG["demand"]) > CONFIG["demand"] * 0.01:
    print(f"WARNING: observed gross {n_obs:,} differs from frozen demand "
          f"{CONFIG['demand']:,} by >1% - check demand provenance.")

observed new-urban cells (AOI-masked): 29,656


# 4. run rasters: Morris results + Phase 0 baselines 

In [5]:

print("loading run rasters (final tick)...")
results_dict  = load_run_dicts(
    mapping = CONFIG["scenario_run"],
    seeds = CONFIG["seeds_results"],
    label = "results (s0 + Morris s_m001-s_m066, seed 11)")
baseline_dict = load_run_dicts(
    mapping = CONFIG["scenario_baseline"],
    seeds = CONFIG["seeds_baseline"],
    label = "baselines (seed ensemble 11/22/33)")

loading run rasters (final tick)...
  loaded expansion_income_t025.asc  <- R125/seed11
  loaded expansion_income_t025.asc  <- R129/seed11
  loaded expansion_income_t025.asc  <- R130/seed11
  loaded expansion_income_t025.asc  <- R131/seed11
  loaded expansion_income_t025.asc  <- R132/seed11
  loaded expansion_income_t025.asc  <- R133/seed11
  loaded expansion_income_t025.asc  <- R134/seed11
  loaded expansion_income_t025.asc  <- R135/seed11
  loaded expansion_income_t025.asc  <- R136/seed11
  loaded expansion_income_t025.asc  <- R137/seed11
  loaded expansion_income_t025.asc  <- R138/seed11
  loaded expansion_income_t025.asc  <- R139/seed11
  loaded expansion_income_t025.asc  <- R140/seed11
  loaded expansion_income_t025.asc  <- R141/seed11
  loaded expansion_income_t025.asc  <- R142/seed11
  loaded expansion_income_t025.asc  <- R143/seed11
  loaded expansion_income_t025.asc  <- R144/seed11
  loaded expansion_income_t025.asc  <- R145/seed11
  loaded expansion_income_t025.asc  <- R146/se

In [ ]:
results_dict

# 5. evaluate per seed: results and baselines SEPARATELY 

In [6]:

fom_results_seeds = evaluate_scenario_fom(
    results_dict = results_dict,
    obs_new = obs_new,
    seeds_list = CONFIG["seeds_results"],
    initial_urban = initial_urban,
    valid_mask = valid_mask,
    tolerances = CONFIG["tolerances"],
    variant = CONFIG["variant"],
    per_tick = "final",
    start_year = CONFIG["calibration_years"][0],
    case_city = CONFIG["case_city"],
    cell_km = CONFIG["cell_km"],
    silent = True)

fom_baseline_seeds = evaluate_scenario_fom(
    results_dict = baseline_dict,
    obs_new = obs_new,
    seeds_list = CONFIG["seeds_baseline"],
    initial_urban = initial_urban,
    valid_mask = valid_mask,
    tolerances = CONFIG["tolerances"],
    variant = CONFIG["variant"],
    per_tick = "final",
    start_year = CONFIG["calibration_years"][0],
    case_city = CONFIG["case_city"],
    cell_km = CONFIG["cell_km"],
    silent = True)
print(f"evaluated: {fom_results_seeds['Scenario'].nunique()} result scenarios, "
      f"{fom_baseline_seeds['Scenario'].nunique()} baseline scenarios")

evaluated: 67 result scenarios, 3 baseline scenarios


# 6. ensemble MEANS on both sides (skill protocol)

In [7]:

fom_results  = aggregate_ensemble(fom_results_seeds)
fom_baseline = aggregate_ensemble(fom_baseline_seeds)
print("baseline ensemble means (should match ti_city_12_2):")
print(fom_baseline[["Scenario", "n_seeds", "N_sim", "FoM"]].to_string(index=False))

baseline ensemble means (should match ti_city_12_2):
   Scenario  n_seeds   N_sim      FoM
     s_edge        3 29653.0 0.334617
   s_random        3 29653.0 0.061010
s_roadbound        3 29656.0 0.132262


# 7. skill vs strongest baseline per metric

In [8]:

fom_skill = normalize_against_baselines(
    fom_table = fom_results,
    baseline_fom_table = fom_baseline,
    perfect = 1.0)

# 8. Summary

In [9]:
# 8a. summary: anchor bar + Morris ranking
fuzzy_cols = [c for c in fom_baseline.columns if c.startswith("FuzzyFoM")]
print("=== BASELINE ANCHOR (ensemble means over seeds 11/22/33) ===")
print(fom_baseline[["Scenario", "N_sim", "FoM"] + fuzzy_cols].to_string(index=False))

s_edge_fom  = float(fom_baseline.loc[fom_baseline["Scenario"] == "s_edge", "FoM"].iloc[0])
s0_fom      = float(fom_results.loc[fom_results["Scenario"] == "s0", "FoM"].iloc[0])
morris      = fom_results[fom_results["Scenario"] != "s0"].copy()
n_beat_edge = int((morris["FoM"] > s_edge_fom).sum())
n_beat_s0   = int((morris["FoM"] > s0_fom).sum())
print(f"\n=== ANCHOR BAR (strict FoM, final tick) ===")
print(f"s_edge ensemble : {s_edge_fom:.3f}   (the bar, Phase 0 anchor)")
print(f"s0 seed-11      : {s0_fom:.3f}")
print(f"Morris runs beating s_edge: {n_beat_edge}/66 | beating s0: {n_beat_s0}/66")

print("\n=== TOP 15 MORRIS RUNS (strict FoM) ===")
top = morris.nlargest(15, "FoM")[["Scenario", "N_sim", "FoM"] + fuzzy_cols]
print(top.to_string(index=False))

skill_cols = ["Scenario", "FoM"] + [c for c in fom_skill.columns if c.endswith("_skill")]
print("\n=== SKILL (vs strongest baseline, per metric) — best 10 by FoM_skill ===")
best_skill = fom_skill[fom_skill["Scenario"] != "s0"].nlargest(10, "FoM_skill")
print(best_skill[skill_cols].to_string(index=False))
print("\nReading: skill > 0 means the run beats EVERY baseline on that metric; "
      "compare the s0 row to the Phase 0 anchor (strict-FoM skill was -0.041).")

=== BASELINE ANCHOR (ensemble means over seeds 11/22/33) ===
   Scenario   N_sim      FoM  FuzzyFoM_t1  FuzzyFoM_t2  FuzzyFoM_t3  FuzzyFoM_t5
     s_edge 29653.0 0.334617     0.526108     0.590067     0.619479     0.645942
   s_random 29653.0 0.061010     0.292864     0.438621     0.488760     0.514145
s_roadbound 29656.0 0.132262     0.405820     0.496917     0.529427     0.550394

=== ANCHOR BAR (strict FoM, final tick) ===
s_edge ensemble : 0.335   (the bar, Phase 0 anchor)
s0 seed-11      : 0.308
Morris runs beating s_edge: 0/66 | beating s0: 2/66

=== TOP 15 MORRIS RUNS (strict FoM) ===
Scenario   N_sim      FoM  FuzzyFoM_t1  FuzzyFoM_t2  FuzzyFoM_t3  FuzzyFoM_t5
  s_m003 28949.0 0.314632     0.452051     0.500011     0.526234     0.555060
  s_m002 28915.0 0.313133     0.449982     0.499081     0.526208     0.556542
  s_m039 27176.0 0.299522     0.422907     0.476551     0.508335     0.544463
  s_m001 28924.0 0.281838     0.410700     0.467593     0.501751     0.540263
  s_m040 27

In [10]:
# 8b. y-metrics table for the Morris sensitivity step -------------------
fuzzy_all = [c for c in fom_results.columns if c.startswith("FuzzyFoM")]
y_metrics = fom_results.merge(
    fom_skill[["Scenario"] + [c for c in fom_skill.columns if c.endswith("_skill")]],
    on = "Scenario", how = "left")
y_metrics["run_num"] = y_metrics["Scenario"].map(
    {s: r for s, r in CONFIG["scenario_run"].items()})
y_metrics = y_metrics.sort_values("run_num").reset_index(drop = True)
print(f"y_metrics: {len(y_metrics)} rows (incl. s0 = R125), "
      f"columns: {list(y_metrics.columns)}")

# optional alignment with the design table from ti_city_13_1 (not required)
design_path = docs / "JAK_TA-City_model_R129_194_design.csv"
if design_path.exists():
    design = pd.read_csv(design_path)
    morris_y = y_metrics[y_metrics["Scenario"] != "s0"].reset_index(drop = True)
    if len(design) == len(morris_y):
        design_metrics = pd.concat(
            [design.reset_index(drop = True),
             morris_y[["FoM"] + fuzzy_all].reset_index(drop = True)], axis = 1)
        print(f"design.csv aligned: {len(design_metrics)} rows, "
              f"{len(design.columns)} design columns + metrics")
    else:
        print(f"WARNING: design.csv has {len(design)} rows, expected {len(morris_y)} - not aligned")
else:
    print(f"note: {design_path.name} not found - run ti_city_13_1 first if you want the aligned table")

y_metrics: 67 rows (incl. s0 = R125), columns: ['Scenario', 'FuzzyFoM_t1', 'FuzzyFoM_t2', 'FuzzyFoM_t3', 'FuzzyFoM_t5', 'FoM', 'N_sim', 'Hits', 'Misses', 'FalseAlarms', 'n_seeds', 'FuzzyFoM_t1_skill', 'FuzzyFoM_t2_skill', 'FuzzyFoM_t3_skill', 'FuzzyFoM_t5_skill', 'FoM_skill', 'run_num']
design.csv aligned: 66 rows, 27 design columns + metrics


In [12]:
# 9. export to docs/ -----------------------------------------------------
exports = {
    f"{case_city}_TA-City_model_R{first_run}_{last_run}_fom_seeds.csv": fom_results_seeds,
    f"{case_city}_TA-City_model_R{first_run}_{last_run}_baseline_fom_seeds.csv": fom_baseline_seeds,
    f"{case_city}_TA-City_model_R{first_run}_{last_run}_fom_table.csv": fom_results,
    f"{case_city}_TA-City_model_R{first_run}_{last_run}_baseline_fom_table.csv": fom_baseline,
    f"{case_city}_TA-City_model_R{first_run}_{last_run}_skill.csv": fom_skill,
    f"{case_city}_TA-City_model_R{first_run}_{last_run}_y_metrics.csv": y_metrics,
}
for name, table in exports.items():
    table.to_csv(docs / name, index = False)
    print(f"written: docs/{name}")

written: docs/JAK_TA-City_model_R129_194_fom_seeds.csv
written: docs/JAK_TA-City_model_R129_194_baseline_fom_seeds.csv
written: docs/JAK_TA-City_model_R129_194_fom_table.csv
written: docs/JAK_TA-City_model_R129_194_baseline_fom_table.csv
written: docs/JAK_TA-City_model_R129_194_skill.csv
written: docs/JAK_TA-City_model_R129_194_y_metrics.csv
